## Objective
Load ImageNet-pretrained DenseNet-121 weights, freeze the entire feature extractor,
and fine-tune only the new dropout classification head. Tests whether ImageNet transfer
learning provides a stronger starting point than training from scratch.
 
## Architecture Changes
 
| Component | 04. Dropout Weighted | 05. Pretrained Weighted |
|---|---|---|
| Pretrained | False | **True (ImageNet)** |
| Frozen backbone | False | **True** |
| Hidden dim | 256 | 256 |
| Head structure | Linear → ReLU → Dropout → Linear | Linear → ReLU → Dropout → Linear |
| Dropout | 0.5 | **0.4** |
| Pos weight | enabled | enabled |
| Optimiser | AdamW | AdamW |
| Weight decay | 1e-4 | 1e-4 |
| LR | 1e-3 | 1e-3 |
| Epochs | 30 | 30 |
 
## Hypothesis
DenseNet-121 pre-trained on ImageNet has already learned hierarchical features (edges,
textures, patterns) relevant to dermoscopic images. Freezing the backbone prevents
catastrophic forgetting of these features and reduces the trainable parameter count
to just the classification head. Dropout is slightly reduced to 0.4 (from 0.5) since
the frozen backbone already acts as a structural constraint on the feature space.

## Import libraries, set seed, and choose device

In [ ]:
import sys
import random
from pathlib import Path
 
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
 
ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / 'src').exists())
sys.path.insert(0, str(ROOT))
 
from src.data.dataset import HAM10000Dataset
from src.data.dataloader import get_dataloaders
from src.data.transform import get_augmented_train_transforms
from src.models.densenet import get_densenet121
from src.training.trainer import train_one_epoch, validate_one_epoch
from src.utils import plot_training_curves, find_best_threshold, evaluate_model
 
import pandas as pd
 
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
 
set_seed(42)
 
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

## Load and split data

In [ ]:
train_dataset = HAM10000Dataset(
    csv_path=str(ROOT / 'data_new/splits/train.csv'),
    image_dir=str(ROOT / 'data_new/images/train'),
    transform=get_augmented_train_transforms(image_size=224),
)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)
 
_, val_loader, test_loader = get_dataloaders(
    train_csv=str(ROOT / 'data_new/splits/train.csv'),
    val_csv=str(ROOT / 'data_new/splits/val.csv'),
    test_csv=str(ROOT / 'data_new/splits/test.csv'),
    image_dir=str(ROOT / 'data_new/images/train'),
    test_image_dir=str(ROOT / 'data_new/images/test'),
    batch_size=32,
    image_size=224,
    num_workers=4,
)
 
train_df     = pd.read_csv(ROOT / 'data_new/splits/train.csv')
num_melanoma = (train_df['label'] == 1).sum()
num_nevus    = (train_df['label'] == 0).sum()
pos_weight   = torch.tensor([num_nevus / num_melanoma], dtype=torch.float32).to(device)
print('Positive weight:', pos_weight)

## Model Definition

In [ ]:
model = get_densenet121(
    num_classes=1,
    pretrained=True,
    freeze_backbone=True,
    dropout=0.4,
    hidden_dim=256,
).to(device)
 
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
 
# Only optimise the trainable classifier head
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
)
 
num_epochs = 30
scheduler  = CosineAnnealingLR(optimizer, T_max=num_epochs)
 
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')
print('Pretrained: True | Frozen: True | Dropout: 0.4 | Hidden dim: 256 | Head: Linear → ReLU → Dropout → Linear')

## Training Loop

In [ ]:
best_val_auc = 0.0
train_history, val_history = [], []
 
for epoch in range(num_epochs):
    train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics   = validate_one_epoch(model, val_loader, criterion, device)
 
    scheduler.step()
 
    train_history.append(train_metrics)
    val_history.append(val_metrics)
 
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"  Train | Loss: {train_metrics['loss']:.4f}, Bal Acc: {train_metrics['balanced_accuracy']:.4f}, Recall: {train_metrics['recall']:.4f}, F2: {train_metrics['f2']:.4f}, AUC: {train_metrics['auc']:.4f}")
    print(f"  Val   | Loss: {val_metrics['loss']:.4f}, Bal Acc: {val_metrics['balanced_accuracy']:.4f}, Recall: {val_metrics['recall']:.4f}, F2: {val_metrics['f2']:.4f}, AUC: {val_metrics['auc']:.4f}")
 
    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        torch.save(model.state_dict(), ROOT / 'models/densenet121_pretrained_weighted_best.pth')
        print(f'  -> Saved best model (val AUC: {best_val_auc:.4f})')

## Plot Train and Validation Curves

In [ ]:
plot_training_curves(train_history, val_history)

## Threshold Tuning (Best Val F2)

In [ ]:
model.load_state_dict(torch.load(str(ROOT / 'models/densenet121_pretrained_weighted_best.pth'), map_location=device))
best_threshold, best_f2 = find_best_threshold(model, val_loader, device)

## Test Set Evaluation

In [ ]:
evaluate_model(model, test_loader, device, threshold=best_threshold)